# FinComplaint AI - pipeline NLP complet sur donnees reelles

SAIM Bahaeddine et OUNI Hazem. Preparation, classification baseline et Transformer, recherche lexicale et semantique, RAG generatif, evaluation et tests. Les effectifs et limites sont explicites. Les donnees 2026 restent reservees. Les resultats d'evaluation ne sont pas une promesse de performance en production.

In [27]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
ROWS_PER_FILE = 3000
REBUILD_DEMO = True
# False conserve les artefacts existants. True reproduit les calculs.
if REBUILD_DEMO:
    subprocess.run(
        [sys.executable, "-m", "scripts.prepare_demo", "--rows-per-file", str(ROWS_PER_FILE)], check=True, cwd=ROOT
    )

CalledProcessError: Command '['c:\\Users\\hazem.LENOVO.000\\Downloads\\FINCOMPLAINT-AI\\.venv\\Scripts\\python.exe', '-m', 'scripts.prepare_demo', '--rows-per-file', '3000']' returned non-zero exit status 1.

# 01 — Préparation et exploration

SAIM Bahaeddine et OUNI Hazem. Corpus : archives CFPB 2025. Ce notebook explore une démonstration réelle limitée aux 3 000 premières lignes de chaque export 2025. Aucun résultat sur le corpus complet ou sur 2026 n’est déduit de cet échantillon. Préparation reproductible : `python -m scripts.prepare_demo`. Pour le corpus complet : `python run_pipeline.py`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Ouvrir depuis la racine ou notebooks/"
sys.path.insert(0, str(ROOT))
import json
import sqlite3
from contextlib import closing

import pandas as pd

REPORTS = ROOT / "reports/final"
DATABASE = ROOT / "data/processed/demo.sqlite"
assert DATABASE.is_file(), "Executer python -m scripts.prepare_demo depuis la racine"

## Origine et sélection
Les récits sont conservés avec leur ID et fichier source. Les dates invalides, textes vides et doublons normalisés sont exclus. La partition 2025 est déterminée par hash (environ 80 % train, 20 % validation). Voir le README pour les conditions de réutilisation.

In [ ]:
quality = json.loads((REPORTS / "data_quality.json").read_text(encoding="utf-8"))
pd.DataFrame([quality["counts"]]).T.rename(columns={0: "effectif"})

,effectif
rows_read,18000
empty_narrative,14110
kept,2521
duplicate_text,1369


In [ ]:
with closing(sqlite3.connect(DATABASE)) as db:
    distribution = pd.read_sql_query("SELECT product, split, COUNT(*) AS n FROM complaints GROUP BY product, split", db)
distribution.pivot(index="product", columns="split", values="n").fillna(0).astype(int)

split,train,validation
product,,
Checking or savings account,111,26
Credit card,106,20
Credit reporting or other personal consumer reports,1336,336
Debt collection,235,62
Debt or credit management,10,3
"Money transfer, virtual currency, or money service",55,17
Mortgage,40,10
"Payday loan, title loan, personal loan, or advance loan",15,6
Prepaid card,18,4


In [ ]:
pd.DataFrame(quality["lengths"])

,split,min,mean,max
0,train,51,1331.238142,35581
1,validation,51,1448.764588,34802


Les textes sont déséquilibrés et les premières lignes des fichiers ne constituent pas un échantillon représentatif. Les quasi-doublons ne sont pas détectés. La longueur et les effectifs par classe doivent guider les expériences suivantes.

# 02 — Classification et comparaison

Baseline entraînée sur les textes train uniquement : TF-IDF uni/bigrammes + régression logistique pondérée. Comparaison avec une prédiction de classe majoritaire, sans ajuster de paramètres sur la validation. DistilBERT reste une extension.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Ouvrir depuis la racine ou notebooks/"
sys.path.insert(0, str(ROOT))
import sqlite3
from contextlib import closing

import pandas as pd

REPORTS = ROOT / "reports/final"
DATABASE = ROOT / "data/processed/demo.sqlite"
assert DATABASE.is_file(), "Executer python -m scripts.prepare_demo depuis la racine"

In [ ]:
import joblib
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, f1_score

model = joblib.load(ROOT / "artifacts/final/baseline.joblib")
with closing(sqlite3.connect(DATABASE)) as db:
    train = pd.read_sql_query(
        "SELECT narrative, product FROM complaints WHERE split='train' ORDER BY text_hash LIMIT 30000", db
    )
    validation = pd.read_sql_query(
        "SELECT narrative, product FROM complaints WHERE split='validation' ORDER BY text_hash LIMIT 10000", db
    )
labels = sorted(set(train["product"]) | set(validation["product"]))
dummy = DummyClassifier(strategy="most_frequent").fit(train[["narrative"]], train["product"])
predictions = model.predict(validation["narrative"])
comparison = pd.DataFrame(
    [
        {
            "modele": "Classe majoritaire",
            "macro_f1": f1_score(
                validation["product"],
                dummy.predict(validation[["narrative"]]),
                labels=labels,
                average="macro",
                zero_division=0,
            ),
        },
        {
            "modele": "TF-IDF + regression logistique",
            "macro_f1": f1_score(validation["product"], predictions, labels=labels, average="macro", zero_division=0),
        },
    ]
)
comparison

,modele,macro_f1
0,Classe majoritaire,0.073338
1,TF-IDF + regression logistique,0.496604


In [ ]:
pd.DataFrame(
    classification_report(validation["product"], predictions, labels=labels, output_dict=True, zero_division=0)
).T

,precision,recall,f1-score,support
Checking or savings account,0.571429,0.923077,0.705882,26.000000
Credit card,0.551724,0.800000,0.653061,20.000000
Credit reporting or other personal consumer reports,0.929936,0.869048,0.898462,336.000000
Debt collection,0.525641,0.661290,0.585714,62.000000
Debt or credit management,1.000000,0.333333,0.500000,3.000000
"Money transfer, virtual currency, or money service",0.800000,0.470588,0.592593,17.000000
Mortgage,0.857143,0.600000,0.705882,10.000000
"Payday loan, title loan, personal loan, or advance loan",0.000000,0.000000,0.000000,6.000000
Prepaid card,0.000000,0.000000,0.000000,4.000000
Student loan,0.333333,0.571429,0.421053,7.000000


## Analyse des erreurs
Les cinq premières erreurs dans l’ordre stable des hash sont affichées sans publier les textes. Une classe sans effectif de validation ne permet pas d’estimer sa performance. La macro-F1 donne le même poids à chaque classe.

In [ ]:
errors = validation.assign(prediction=predictions)
errors.loc[errors["product"] != errors["prediction"], ["product", "prediction"]].head()

,product,prediction
3,Credit reporting or other personal consumer re...,Debt collection
6,"Money transfer, virtual currency, or money ser...",Checking or savings account
12,Credit reporting or other personal consumer re...,Debt collection
18,Debt collection,Credit reporting or other personal consumer re...
33,Debt or credit management,Credit reporting or other personal consumer re...


Le modèle Joblib doit provenir d’une source de confiance. Le test temporel 2026 reste fermé pendant ces expériences. La reproductibilité dépend des versions consignées dans manifest.json.

# 03 — Recherche et RAG

Recherche lexicale TF-IDF avec similarité cosinus sur le train 2025. Le mode par défaut retourne des extraits avec leurs sources ; il ne réalise pas de génération. Une synthèse par modèle local Ollama peut être activée explicitement. Aucun embedding ou modèle externe n’est téléchargé par ce notebook.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Ouvrir depuis la racine ou notebooks/"
sys.path.insert(0, str(ROOT))
import pandas as pd

REPORTS = ROOT / "reports/final"
DATABASE = ROOT / "data/processed/demo.sqlite"
assert DATABASE.is_file(), "Executer python -m scripts.prepare_demo depuis la racine"

In [ ]:
import joblib

from src.rag import ask
from src.retriever import search

index = joblib.load(ROOT / "artifacts/final/retriever.joblib")
question = "credit report identity theft"
# Comparer k=3 et k=5 : ces scores sont des similarites, pas des probabilites.
experiments = []
for k in [3, 5]:
    for source in search(index, question, k):
        experiments.append({"k": k, **{field: source[field] for field in ["complaint_id", "source_file", "score"]}})
pd.DataFrame(experiments)

,k,complaint_id,source_file,score
0,3,13293607,CCDB_Export_11_May_2025_through_June_2025.csv,0.314390
1,3,15651725,CCDB_Export_13_September_2025_through_October_...,0.253114
2,3,13260546,CCDB_Export_11_May_2025_through_June_2025.csv,0.238306
3,5,13293607,CCDB_Export_11_May_2025_through_June_2025.csv,0.314390
4,5,15651725,CCDB_Export_13_September_2025_through_October_...,0.253114
5,5,13260546,CCDB_Export_11_May_2025_through_June_2025.csv,0.238306
6,5,12239958,CCDB_Export_10_March_2025_through_April_2025.csv,0.223788
7,5,11599301,CCDB_Export_9_January_2025_through_February_20...,0.223534


In [ ]:
answer = ask(index, question)
# Apercu bref; les extraits complets sont consultables avec la CLI.
print(answer["mode"])
print(answer["answer"][:240])
print("Sources :", [source["complaint_id"] for source in answer["sources"]])

extractive
[13293607] I am writing to urgently request the removal of fraudulent accounts on my credit report due to identity theft. The inaccurate information is causing significant harm to my credit, and I expect immediate corrective action.

[15651
Sources : ['13293607', '15651725', '13260546']


In [ ]:
ask(index, "quasar nebula astrophysics")

{'mode': 'abstention',
 'answer': 'Aucun passage pertinent trouve dans cet index.',
 'sources': [],
 'citations_valid': True}

## Génération locale facultative
Après installation d’Ollama et d’un modèle compatible, définir `OLLAMA_MODEL` dans le terminal puis appeler `python -m src.demo ask "credit report identity theft" --artifacts artifacts/final --generate`. La requête envoyée au modèle contient la question et les extraits sourcés. Le contrôle des IDs cités ne garantit pas la fidélité des affirmations. La génération Ollama n’est pas exécutée ni évaluée ici. [Documentation du protocole](https://docs.ollama.com/api/chat).

L’abstention est déclenchée uniquement en l’absence de recouvrement lexical. Une question hors domaine partageant des mots du corpus peut donc produire des résultats. Comparer k=3 et k=5 ne mesure pas leur pertinence : il faut des jugements humains indépendants.

# 04 — Évaluation et preuves de fonctionnement

Cette évaluation sépare les scores de classification sur validation et les contrôles fonctionnels de traçabilité. Aucun score de pertinence ou de fidélité humaine n’est inventé.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Ouvrir depuis la racine ou notebooks/"
sys.path.insert(0, str(ROOT))
import json
import sqlite3
from contextlib import closing

import pandas as pd

REPORTS = ROOT / "reports/final"
DATABASE = ROOT / "data/processed/demo.sqlite"
assert DATABASE.is_file(), "Executer python -m scripts.prepare_demo depuis la racine"

In [ ]:
metrics = json.loads((REPORTS / "baseline_metrics.json").read_text(encoding="utf-8"))
pd.DataFrame(
    [
        {"partition": split, "effectif": score["rows"], "macro_f1": score.get("macro_f1")}
        for split, score in metrics["splits"].items()
    ]
)

,partition,effectif,macro_f1
0,validation,497,0.496604


In [ ]:
pd.read_csv(REPORTS / "confusion_validation.csv", index_col=0)

,Checking or savings account,Credit card,Credit reporting or other personal consumer reports,Debt collection,Debt or credit management,"Money transfer, virtual currency, or money service",Mortgage,"Payday loan, title loan, personal loan, or advance loan",Prepaid card,Student loan,Vehicle loan or lease
Checking or savings account,24,2,0,0,0,0,0,0,0,0,0
Credit card,1,16,2,0,0,1,0,0,0,0,0
Credit reporting or other personal consumer reports,0,6,292,31,0,0,0,0,0,7,0
Debt collection,1,3,15,41,0,0,1,0,0,1,0
Debt or credit management,0,0,1,1,1,0,0,0,0,0,0
"Money transfer, virtual currency, or money service",7,1,0,0,0,8,0,0,0,0,1
Mortgage,2,0,1,1,0,0,6,0,0,0,0
"Payday loan, title loan, personal loan, or advance loan",3,0,0,2,0,0,0,0,0,0,1
Prepaid card,2,1,0,0,0,1,0,0,0,0,0
Student loan,2,0,1,0,0,0,0,0,0,4,0


## Contrôles fonctionnels
Vérifier que chaque extrait provient du récit stocké et que ses métadonnées correspondent au corpus. Ces assertions évaluent la traçabilité du mode extractif, sans évaluer la compréhension ou la qualité d’une synthèse.

In [ ]:
import joblib

from src.rag import ask

index = joblib.load(ROOT / "artifacts/final/retriever.joblib")
checks = []
with closing(sqlite3.connect(DATABASE)) as db:
    for question in ["credit report identity theft", "mortgage payment escrow"]:
        result = ask(index, question)
        assert result["sources"]
        for source in result["sources"]:
            row = db.execute(
                "SELECT narrative, source_file, split FROM complaints WHERE complaint_id=?", (source["complaint_id"],)
            ).fetchone()
            assert row and row[0].startswith(source["excerpt"])
            assert row[1] == source["source_file"] and row[2] == "train"
        checks.append({"question": question, "sources_verifiees": len(result["sources"])})
assert ask(index, "quasar nebula astrophysics")["mode"] == "abstention"
assert not metrics["test_evaluated"]
pd.DataFrame(checks)

,question,sources_verifiees
0,credit report identity theft,3
1,mortgage payment escrow,3


In [ ]:
manifest = json.loads((REPORTS / "manifest.json").read_text(encoding="utf-8"))
{
    "scope": manifest["scope"],
    "python": manifest["python"],
    "packages": manifest["packages"],
    "test_2026_opened": manifest["test_2026_opened"],
}

{'scope': 'Demonstration sur les premieres lignes de chaque export 2025, non representatif',
 'python': '3.12.3',
 'packages': {'pandas': '2.3.1', 'scikit-learn': '1.9.0', 'joblib': '1.5.3'},
 'test_2026_opened': False}

# Experience complete : spaCy, DistilBERT, FAISS et RAG generatif

Les donnees sont reelles. Les limites CPU sont explicites et identiques pour la comparaison de classification. Adaptation supervisee de la tete DistilBERT : encodeur pretrained gele, pas de fine-tuning integral. Le test 2026 reste ferme. La couverture fonctionnelle ne garantit pas la qualite scientifique d'une synthese.

In [ ]:
import os

os.environ["HF_HOME"] = str(ROOT / ".models")
os.environ["HF_HUB_DISABLE_XET"] = "1"
import torch

torch.set_num_threads(4)
import spacy

from src.advanced import SemanticIndex, adapt_distilbert, answer_question, ranking_metrics

nlp = spacy.load("en_core_web_sm")
doc = nlp(train.narrative.iloc[0][:2000])
display(pd.DataFrame([(t.text, t.lemma_, t.pos_) for t in doc][:30], columns=["token", "lemme", "categorie"]))
print("Entites:", [(e.text, e.label_) for e in doc.ents])

,token,lemme,categorie
0,This,this,DET
1,morning,morning,NOUN
2,",",",",PUNCT
3,I,I,PRON
4,noticed,notice,VERB
5,incorrect,incorrect,ADJ
6,accounts,account,NOUN
7,appearing,appear,VERB
8,on,on,ADP
9,my,my,PRON


Entites: [('This morning', 'TIME')]


## Comparaison supervisee sur les memes effectifs
Maximum 24 textes par classe en apprentissage et 8 par classe en validation, choisis par l'ordre deterministe des hash deja etabli. Aucun parametre ne sera choisi sur 2026. Les textes du Transformer sont tronques a 128 tokens ; TF-IDF utilise les textes complets. Cette difference est une limite de comparaison.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

small_train = train.groupby("product", group_keys=False).head(24).copy()
small_validation = validation.groupby("product", group_keys=False).head(8).copy()
print("Train / validation:", len(small_train), len(small_validation))
baseline_small = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=500, class_weight="balanced", random_state=42)
)
baseline_small.fit(small_train.narrative, small_train["product"])
advanced_predictions, history = adapt_distilbert(small_train, small_validation, epochs=30)
common_labels = sorted(set(small_train["product"]) | set(small_validation["product"]))
model_comparison = pd.DataFrame(
    [
        {
            "modele": name,
            "macro_f1": f1_score(
                small_validation["product"], pred, labels=common_labels, average="macro", zero_division=0
            ),
        }
        for name, pred in [
            ("TF-IDF memes donnees", baseline_small.predict(small_validation.narrative)),
            ("DistilBERT tete adaptee", advanced_predictions),
        ]
    ]
)
display(model_comparison)
display(history.iloc[[0, -1]])
display(
    pd.DataFrame(
        classification_report(
            small_validation["product"], advanced_predictions, labels=common_labels, output_dict=True, zero_division=0
        )
    ).T
)
model_comparison.to_csv(REPORTS / "advanced_classification.csv", index=False)

Train / validation: 235 74


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at <PROJECT_ROOT>\.models\distilbert and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


,modele,macro_f1
0,TF-IDF memes donnees,0.560880
1,DistilBERT tete adaptee,0.402459


,epoch,training_loss
0,1,2.403884
29,30,1.057181


,precision,recall,f1-score,support
Checking or savings account,0.600000,0.750000,0.666667,8.000000
Credit card,0.272727,0.375000,0.315789,8.000000
Credit reporting or other personal consumer reports,0.500000,0.250000,0.333333,8.000000
Debt collection,0.300000,0.375000,0.333333,8.000000
Debt or credit management,0.000000,0.000000,0.000000,3.000000
"Money transfer, virtual currency, or money service",0.583333,0.875000,0.700000,8.000000
Mortgage,0.666667,0.500000,0.571429,8.000000
"Payday loan, title loan, personal loan, or advance loan",0.000000,0.000000,0.000000,6.000000
Prepaid card,0.666667,0.500000,0.571429,4.000000
Student loan,0.571429,0.571429,0.571429,7.000000


## Recherche : decoupage, MiniLM et FAISS
Index limite aux 300 premiers documents train par hash, segments de 80 mots avec recouvrement de 15 mots. MiniLM produit des embeddings normalises ; FAISS IndexFlatIP calcule le cosinus. L'index lexical est compare sur les memes documents. Les requetes sont des exemples de demonstration, pas un benchmark humain independant.

In [ ]:
from src.retriever import build as build_lexical
from src.retriever import search as search_lexical

with closing(sqlite3.connect(DATABASE)) as db:
    records = pd.read_sql_query(
        "SELECT complaint_id, narrative, product, source_file, "
        "date_received FROM complaints WHERE split='train' ORDER BY text_hash LIMIT 300",
        db,
    )
semantic = SemanticIndex(records.to_dict("records"))
lexical = build_lexical(DATABASE, ROOT / "artifacts/final/lexical300.joblib", limit=300)
print("Documents:", len(records), "segments:", len(semantic.passages))
questions = [
    "credit report identity theft",
    "mortgage payment escrow",
    "credit card unauthorized charge",
    "quasar nebula astrophysics",
]
retrieval_comparison = []
for question in questions:
    for k in [3, 5]:
        for name, hits in [
            ("TF-IDF", search_lexical(lexical, question, k)),
            ("MiniLM-FAISS", semantic.search(question, k)),
        ]:
            for hit in hits:
                retrieval_comparison.append(
                    {"question": question, "k": k, "engine": name, "id": hit["complaint_id"], "score": hit["score"]}
                )
display(pd.DataFrame(retrieval_comparison))
import faiss

faiss.write_index(semantic.index, str(ROOT / "artifacts/final/semantic.faiss"))
pd.DataFrame(semantic.passages).to_json(ROOT / "artifacts/final/passages.json", orient="records")

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

Documents: 300 segments: 1311


,question,k,engine,id,score
0,credit report identity theft,3,TF-IDF,13283458,0.350238
1,credit report identity theft,3,TF-IDF,13263754,0.350199
2,credit report identity theft,3,TF-IDF,12239780,0.322920
3,credit report identity theft,3,MiniLM-FAISS,16968695,0.762023
4,credit report identity theft,3,MiniLM-FAISS,13283458,0.734205
5,credit report identity theft,3,MiniLM-FAISS,11344549,0.730551
6,credit report identity theft,5,TF-IDF,13283458,0.350238
7,credit report identity theft,5,TF-IDF,13263754,0.350199
8,credit report identity theft,5,TF-IDF,12239780,0.322920
9,credit report identity theft,5,TF-IDF,15633796,0.321843


## Evaluation de recherche : controle automatique et annotation humaine
Le controle ci-dessous utilise 12 questions construites a partir d'un extrait de document indexe (known-item retrieval). Il mesure la recuperation de l'element connu, sans estimer la pertinence des questions utilisateurs. Les annotations humaines independantes restent necessaires pour un benchmark metier.

In [ ]:
retrieval_scores = []
for record in records.head(12).to_dict("records"):
    query = " ".join(record["narrative"].split()[:20])
    for k in [3, 5]:
        for name, hits in [("TF-IDF", search_lexical(lexical, query, k)), ("MiniLM-FAISS", semantic.search(query, k))]:
            ids = list(dict.fromkeys(str(h["complaint_id"]) for h in hits))
            retrieval_scores.append({"engine": name, "k": k, **ranking_metrics(ids, {str(record["complaint_id"])})})
retrieval_summary = pd.DataFrame(retrieval_scores).groupby(["engine", "k"]).mean()
display(retrieval_summary)
retrieval_summary.to_csv(REPORTS / "known_item_retrieval.csv")

recall       mrr
engine       k                  
MiniLM-FAISS 3     1.0  0.916667
             5     1.0  0.916667
TF-IDF       3     1.0  1.000000
             5     1.0  1.000000

## RAG generatif local avec preuves consultables
FLAN-T5-small genere une reponse conditionnee sur les extraits retrouves. Comparaison top-3/top-5. Seuil cosinus fixe a priori a 0,35 pour l'abstention ; ce seuil exploratoire n'est pas calibre. Les IDs cites sont controles, mais cela ne garantit pas la fidelite. Les citations absentes ou invalides sont signalees : les sources recuperees sont toujours affichees separement.

In [ ]:
rag_results = []
for question in [
    "What problems are reported about mortgage escrow payments?",
    "Who won the football world cup?",
    "How do I bake a chocolate cake?",
]:
    for k in [3, 5]:
        result = answer_question(semantic, question, k=k)
        print("QUESTION:", question, "k=", k)
        print("MODE:", result["mode"], "REPONSE:", result["answer"])
        print("CITATIONS VALIDES:", result["citations_valid"])
        display(pd.DataFrame(result["sources"]))
        rag_results.append({"question": question, "k": k, **result})
(REPORTS / "generated_examples.json").write_text(
    json.dumps(rag_results, ensure_ascii=False, indent=2), encoding="utf-8"
)
display(pd.DataFrame([{key: value for key, value in r.items() if key != "sources"} for r in rag_results]))

QUESTION: What problems are reported about mortgage escrow payments? k= 3
MODE: abstention_invalid_citations REPONSE: The generated draft has no valid citations; inspect the retrieved evidence.
CITATIONS VALIDES: False


,complaint_id,product,source_file,date_received,excerpt,offset_words,score
0,11344920,Mortgage,CCDB_Export_9_January_2025_through_February_20...,2025-01-01,"company is currently closed, and I am unable t...",65,0.672423
1,11344920,Mortgage,CCDB_Export_9_January_2025_through_February_20...,2025-01-01,On the XXXX of XX/XX/year> my loan servicing c...,0,0.470985
2,11345147,Credit reporting or other personal consumer re...,CCDB_Export_9_January_2025_through_February_20...,2025-01-01,back and forth with a customer service rep who...,130,0.460436


QUESTION: What problems are reported about mortgage escrow payments? k= 5
MODE: flan-t5 REPONSE: [11344920] company is currently closed, and I am unable to get this resolved but am doing this as a precaution as my escrow account is now in the negative by - $6900.00. This will affect me negatively. I only realized this when I went online to make our monthly mortgage payment yesterday XX/XX/year>, and noticed this egregious mistake. I will be contacting them tomorrow
CITATIONS VALIDES: True


,complaint_id,product,source_file,date_received,excerpt,offset_words,score
0,11344920,Mortgage,CCDB_Export_9_January_2025_through_February_20...,2025-01-01,"company is currently closed, and I am unable t...",65,0.672423
1,11344920,Mortgage,CCDB_Export_9_January_2025_through_February_20...,2025-01-01,On the XXXX of XX/XX/year> my loan servicing c...,0,0.470985
2,11345147,Credit reporting or other personal consumer re...,CCDB_Export_9_January_2025_through_February_20...,2025-01-01,back and forth with a customer service rep who...,130,0.460436
3,13260809,Checking or savings account,CCDB_Export_11_May_2025_through_June_2025.csv,2025-05-01,representative I spoke with was dismissive and...,195,0.457695
4,12561293,Credit reporting or other personal consumer re...,CCDB_Export_10_March_2025_through_April_2025.csv,2025-03-19,"If I do not receive a satisfactory resolution,...",325,0.439729


QUESTION: Who won the football world cup? k= 3
MODE: abstention REPONSE: Insufficient evidence in this corpus.
CITATIONS VALIDES: True


""


... (3 outputs left out)

""


QUESTION: How do I bake a chocolate cake? k= 5
MODE: abstention REPONSE: Insufficient evidence in this corpus.
CITATIONS VALIDES: True


""


,question,k,mode,answer,generated_draft,citations_valid,warning
0,What problems are reported about mortgage escr...,3,abstention_invalid_citations,The generated draft has no valid citations; in...,egregious mistake,False,Draft withheld because its references could no...
1,What problems are reported about mortgage escr...,5,flan-t5,"[11344920] company is currently closed, and I ...",NaN,True,Retrieved evidence is shown separately. Verify...
2,Who won the football world cup?,3,abstention,Insufficient evidence in this corpus.,NaN,True,NaN
3,Who won the football world cup?,5,abstention,Insufficient evidence in this corpus.,NaN,True,NaN
4,How do I bake a chocolate cake?,3,abstention,Insufficient evidence in this corpus.,NaN,True,NaN
5,How do I bake a chocolate cake?,5,abstention,Insufficient evidence in this corpus.,NaN,True,NaN


## Analyse critique des modeles avances
Un encodeur gele est une adaptation legere economique mais moins expressive qu'un fine-tuning complet. Les classes rares, textes tronques et biais de l'echantillon limitent les resultats. Le petit generateur peut halluciner, repondre a cote ou omettre ses citations ; ces erreurs font partie des resultats, pas d'une panne a masquer. Une evaluation humaine de chaque affirmation est necessaire avant tout usage. Aucun resultat final 2026 n'est annonce.

## Évaluation finale à réaliser
Constituer un jeu de questions indépendant avec sources pertinentes annotées avant la sélection du moteur. Rapporter Recall@3, Recall@5 et MRR, puis vérifier manuellement chaque affirmation et citation des synthèses générées. Ouvrir le test 2026 seulement après fixation des choix : `python -m src.classifier --evaluate-test`. Les résultats de démonstration ne remplacent pas cette évaluation finale.

# Qualite du code et tests

Le code metier reste dans src ; le notebook orchestre et analyse. pytest execute les tests unitaires existants ainsi qu'une integration CSV -> SQLite -> modele -> sources. La couverture n'est pas une mesure de qualite NLP.

In [ ]:
from scripts.verify_project import verify

verify(ROOT)

All checks passed!
45 files already formatted
.......................................                                  [100%]
=============================== tests coverage ================================
_______________ coverage: platform win32, python 3.12.3-final-0 _______________

Name                   Stmts   Miss Branch BrPart  Cover   Missing
------------------------------------------------------------------
src\__init__.py            0      0      0      0   100%
src\advanced.py           98      0     24      0   100%
src\api.py                44      0      0      0   100%
src\classifier.py         51      0     10      0   100%
src\demo.py               30      0      6      0   100%
src\ingestion.py          88      0     30      0   100%
src\preprocessing.py       7      0      0      0   100%
src\rag.py                22      0      6      0   100%
src\retriever.py          25      0      6      0   100%
------------------------------------------------------------------

# Limites et reproductibilite

La validation sur cet echantillon de convenance ne mesure pas le corpus complet. Les valeurs affichees sont recalculees ; elles peuvent differer des rapports du depot. Le test 2026, le benchmark de recherche annote et l'evaluation humaine du RAG ne sont pas revendiques. Pour traiter tous les exports : `python run_pipeline.py`.

Usage IA : notebook d'orchestration et tests complementaires prepares avec Codex ; les auteurs doivent verifier et interpreter les resultats avant remise. Aucun push effectue.

# Test temporel final 2026 - echantillon borne

Les choix precedents sont maintenant fixes. La baseline conservee est evaluee sans reentrainement sur les 3 000 premieres lignes de chaque export 2026, apres filtrage et exclusion des doublons connus avec 2025. Au maximum 1 000 recits sont retenus par ordre de hash. Cette section ouvre explicitement le test temporel, contrairement aux sections de demonstration historique ci-dessus. Le resultat ne mesure pas l'ensemble de 2026.

In [ ]:
from scripts.evaluate_temporal import main as evaluate_temporal

evaluate_temporal()
temporal = json.loads((REPORTS / "temporal_2026.json").read_text(encoding="utf-8"))
display(pd.DataFrame(temporal["classification_report"]).T)
assert not temporal["refitted"]

{
  "model": "fixed baseline trained on 2025 demo",
  "scope": "first 3000 rows per 2026 export; at most 1000 retained hash-ordered narratives",
  "rows_read": 21000,
  "rows_evaluated": 1000,
  "refitted": false,
  "macro_f1": 0.4722267467070354,
  "unseen_labels": [],
  "limitations": "Convenience sample, no near-duplicate audit; not a full-corpus benchmark."
}


,precision,recall,f1-score,support
Checking or savings account,0.442922,0.808333,0.572271,120.00
Credit card,0.657143,0.737968,0.695214,187.00
Credit reporting or other personal consumer reports,0.110000,0.916667,0.196429,12.00
Debt collection,0.867925,0.684524,0.765391,336.00
Debt or credit management,0.000000,0.000000,0.000000,19.00
"Money transfer, virtual currency, or money service",0.613636,0.337500,0.435484,80.00
Mortgage,0.891892,0.825000,0.857143,80.00
"Payday loan, title loan, personal loan, or advance loan",0.000000,0.000000,0.000000,47.00
Prepaid card,1.000000,0.117647,0.210526,17.00
Student loan,0.812500,0.764706,0.787879,51.00


# Demonstrateur REST

Lancer `python -m uvicorn src.api:app --host 127.0.0.1 --port 8011`. Ouvrir http://127.0.0.1:8011/docs. `/classify` utilise la baseline retenue ; `/search` interroge FAISS ; `/ask` genere avec FLAN-T5 et refuse les brouillons sans citations valides. Les tests de contrats HTTP utilisent une injection de dependances ; le notebook execute separement les vrais modeles.